# NoteWeaver: Generate, Review, and Save Notes

This English walkthrough connects the real PDF reader to note generation,
editing, approval, and verified persistence. Follow [setup instructions](../docs/GROQ_SETUP.md).

The default mode is **offline**, which uses fixed test content. To obtain a real
AI answer, change `MODE` to `"groq"`. Only Groq mode sends your selected lecture
page and question to an external service. Your Groq account's plan and quotas
apply; the package never switches to a paid provider automatically.

Keep this notebook inside the checkout. Committed outputs are cleared. The
notebook creates a unique folder under ignored `generated/groq-notebook/`.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
import tempfile
from IPython.display import Image, display
from noteweaver import (
    ContractError, FakeNoteProvider, GenerationError, GroqProvider,
    approve_note, create_project, generate_note, load_project, save_project, update_note,
)
from noteweaver.pdf_reader import PDFReader

MODE = "offline"  # Change to "groq" to make a real API request.
QUESTION = "Why can a free choice still have an opportunity cost?"
LANGUAGE = "en"  # Use "zh-Hant" for Traditional Chinese output.
ORIGINAL_PAGE = 2

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "examples/data/sample_lecture_3p.pdf").is_file()
             and (p / "src/noteweaver/workflow.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook inside the NoteWeaver checkout.")
source = root / "examples/data/sample_lecture_3p.pdf"
output_root = root / "generated/groq-notebook"
output_root.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=output_root))
reader = PDFReader(preview_dir=run_dir / "previews")
project = create_project(reader.inspect(source), source)
latest_note_id = None

## 1. Select and inspect the original lecture page

The demo uses the real bilingual lecture fixture. Change `source` to your own
text-based PDF when ready. Only the selected page is sent in this example.

In [ ]:
page = reader.read_page(source, ORIGINAL_PAGE)
print(f"Original page {page.original_page}: {page.extraction_status}")
print(page.text)
if page.preview_path is not None:
    display(Image(filename=page.preview_path, width=800))
else:
    print("Preview unavailable; extracted text remains available.")

## 2. Choose the provider

In Groq mode, use `GROQ_API_KEY` from your environment or enter the key in the
hidden prompt below. Never put the key in a code cell or commit it. Restarting
the kernel clears the provider object and any key entered through this prompt.
Do not share kernel sessions containing credentials.

In [ ]:
provider = None
if MODE == "offline":
    provider = FakeNoteProvider()
    print("OFFLINE: fixed test content; no API call will be made.")
elif MODE == "groq":
    if os.environ.get("GROQ_API_KEY"):
        provider = GroqProvider()
    else:
        entered_key = getpass("Enter your Groq API key (hidden): ")
        if entered_key.strip():
            provider = GroqProvider(api_key=entered_key)
        else:
            print("No key entered. Generation is disabled.")
        del entered_key
    if provider is not None:
        print("Groq selected. The next cell sends the page text and question.")
else:
    raise ValueError("MODE must be 'offline' or 'groq'.")

## 3. Generate a draft

Run this cell to generate. In Groq mode, every execution makes a new request and
uses quota. No automatic retries or fallback calls are made. A successful call
appends a new draft and its question record; it does not replace previous notes.
On failure, existing project data remains unchanged.

In [ ]:
latest_note_id = None
if provider is None:
    print("Configure a provider in the previous cell first.")
else:
    print("Generating...")
    try:
        updated = generate_note(
            project, original_page=ORIGINAL_PAGE, question=QUESTION,
            provider=provider, language=LANGUAGE, style="example",
        )
    except (ContractError, GenerationError, OSError) as error:
        print(f"Generation failed: {error}")
        print("Existing notes are unchanged. Fix the issue and rerun to retry.")
    else:
        project = updated
        note = project.notes[-1]
        latest_note_id = note.note_id
        print(f"Draft created: {note.title}")
        for section in note.sections:
            print(f"\n[{section.origin} / {section.kind}]\n{section.content}")

## 4. Optionally edit the note

Set `REVISED_CONTENT` to your own explanation, then run this cell. Replacing the
body labels it as user-authored and clears its previous source claims. Editing
resets approval to draft while keeping the note ID and original-page anchor.

In [ ]:
REVISED_CONTENT = None  # Replace with a string to edit the entire note body.
if latest_note_id is None:
    print("No new draft is available.")
elif REVISED_CONTENT is not None:
    project = update_note(project, latest_note_id, content=REVISED_CONTENT)
    print("Edited; the note is a draft and needs review.")
else:
    print("No edit requested.")
if latest_note_id is not None:
    note = next(n for n in project.notes if n.note_id == latest_note_id)
    print(f"\nReview: {note.title} | Status: {note.status}")
    for section in note.sections:
        print(f"\n[{section.origin}] {section.content}")

## 5. Confirm only after reviewing

Change `CONFIRM_REVIEW` to `True` only after reading the displayed note.
Approval records your decision, not a factual correctness guarantee. The
notebook defaults to leaving generated content as a draft, including Run All.

In [ ]:
CONFIRM_REVIEW = False
if CONFIRM_REVIEW and latest_note_id is not None:
    project = approve_note(project, latest_note_id)
    print("Note approved.")
else:
    print("No approval recorded; current review state is preserved.")

## 6. Save and reopen your project

Both drafts and approved notes can be saved. Every execution saves to a new file
in this run folder; no existing project is overwritten. The source PDF must
remain available for verified reopening. This saves JSON, not a study PDF.

In [ ]:
from uuid import uuid4
if project.notes:
    project_file = save_project(project, run_dir / f"study-{uuid4().hex[:8]}.json")
    reopened = load_project(project_file)
    assert reopened == project
    print("Saved and verified:", project_file.relative_to(root))
    print("Note states:", [n.status for n in reopened.notes])
    print("Questions, note IDs, content, and original-page anchors preserved.")
else:
    print("No notes to save yet. Complete generation first.")

## Next steps

The same public `generate_note()` operation is usable from any Python script.
Use `context_pages=(1, 2, 3)` if you explicitly want to send neighboring pages as
well; these must all contain successfully extracted text. An empty/scanned page
needs manual notes or another extraction solution; this demo does not perform OCR.

AI generation requires a real provider; offline output is only an integration
test double. The separate export API or examples/export_demo.py can export approved notes.